In [8]:
import os
import pandas as pd
import requests
import joblib
from dotenv import load_dotenv

load_dotenv() 

FOOTBALL_DATA_API_KEY = os.environ["FOOTBALL_DATA_API_KEY"]
CURRENT_SEASON = "2026"

features = pd.read_csv("../data/features/match_features_multi_season.csv", parse_dates=["utcDate"])

clean_matches = pd.read_csv("../data/processed/clean_matches_multi_season.csv", parse_dates=["utcDate"])
clean_matches = clean_matches.sort_values("utcDate").reset_index(drop=True)

model = joblib.load("../data/features/logistic_regression_model.pkl")

def fetch_scheduled_fixtures():
    url = "https://api.football-data.org/v4/competitions/PL/matches"
    headers = {"X-Auth-Token": FOOTBALL_DATA_API_KEY}
    params = {"status": "SCHEDULED"}

    response = requests.get(url, headers=headers, params=params, timeout=30)
    response.raise_for_status()
    matches = response.json()["matches"]

    fixtures = pd.DataFrame([{
        "match_id": m["id"],
        "utcDate": m["utcDate"],
        "matchday": m["matchday"],
        "season": m["season"]["startDate"][:4],
        "home_team": m["homeTeam"]["name"],
        "away_team": m["awayTeam"]["name"],
    } for m in matches])

    fixtures["utcDate"] = pd.to_datetime(fixtures["utcDate"])
    fixtures = fixtures[fixtures["season"] == CURRENT_SEASON]
    fixtures = fixtures.sort_values("utcDate").reset_index(drop=True)
    return fixtures

remaining_fixtures = fetch_scheduled_fixtures()
print(f"Remaining fixtures this season: {len(remaining_fixtures)}")
remaining_fixtures.head()

Remaining fixtures this season: 352


,match_id,utcDate,matchday,season,home_team,away_team
0,560567,2026-09-06 13:00:00+00:00,3,2026,Everton FC,Manchester United FC
1,560570,2026-09-06 15:30:00+00:00,3,2026,Arsenal FC,Chelsea FC
2,560572,2026-09-12 14:00:00+00:00,4,2026,Crystal Palace FC,Ipswich Town FC
3,560573,2026-09-12 14:00:00+00:00,4,2026,Liverpool FC,Fulham FC
4,560574,2026-09-12 14:00:00+00:00,4,2026,Aston Villa FC,Nottingham Forest FC


In [14]:
from collections import defaultdict

CURRENT_SEASON_INT = int(CURRENT_SEASON)

# --- Restrict to this season's real, already-played matches ---
def fetch_finished_matches_this_season():
    url = "https://api.football-data.org/v4/competitions/PL/matches"
    headers = {"X-Auth-Token": FOOTBALL_DATA_API_KEY}
    params = {"season": CURRENT_SEASON, "status": "FINISHED"}

    response = requests.get(url, headers=headers, params=params, timeout=30)
    response.raise_for_status()
    matches = response.json()["matches"]

    df = pd.DataFrame([{
        "id": m["id"],
        "utcDate": m["utcDate"],
        "home_team_name": m["homeTeam"]["name"],
        "away_team_name": m["awayTeam"]["name"],
        "home_goals": m["score"]["fullTime"]["home"],
        "away_goals": m["score"]["fullTime"]["away"],
        "winner": m["score"]["winner"],
    } for m in matches])

    df["utcDate"] = pd.to_datetime(df["utcDate"])
    return df.sort_values("utcDate").reset_index(drop=True)

season_matches = fetch_finished_matches_this_season()

def build_team_matches(df):
    """Same home/away melt as 04_feature_engineering.ipynb, scoped to one season's real matches."""
    home_rows = df[["utcDate", "id", "home_team_name", "home_goals", "away_goals", "winner"]].copy()
    home_rows.columns = ["utcDate", "match_id", "team", "goals_scored", "goals_conceded", "winner"]
    home_rows["is_home"] = True
    home_rows["result"] = home_rows["winner"].map({"HOME_TEAM": "WIN", "AWAY_TEAM": "LOSS", "DRAW": "DRAW"})

    away_rows = df[["utcDate", "id", "away_team_name", "away_goals", "home_goals", "winner"]].copy()
    away_rows.columns = ["utcDate", "match_id", "team", "goals_scored", "goals_conceded", "winner"]
    away_rows["is_home"] = False
    away_rows["result"] = away_rows["winner"].map({"HOME_TEAM": "LOSS", "AWAY_TEAM": "WIN", "DRAW": "DRAW"})

    tm = pd.concat([home_rows, away_rows]).sort_values(["team", "utcDate"]).reset_index(drop=True)
    tm["points"] = tm["result"].map({"WIN": 3, "DRAW": 1, "LOSS": 0})
    tm["clean_sheet"] = (tm["goals_conceded"] == 0).astype(int)
    tm["is_win"] = (tm["result"] == "WIN").astype(int)
    return tm

team_matches_season = build_team_matches(season_matches)

def init_team_state(team):
    rows = team_matches_season[team_matches_season["team"] == team].sort_values("utcDate")

    state = {
        "recent_points": list(rows["points"].tail(5)),          # for rolling_form_pts
        "win_streak": 0,
        "loss_streak": 0,
        "goals_scored_sum": rows["goals_scored"].sum(),
        "goals_scored_count": len(rows),
        "goals_conceded_sum": rows["goals_conceded"].sum(),
        "goals_conceded_count": len(rows),
        "clean_sheet_sum": rows["clean_sheet"].sum(),
        "clean_sheet_count": len(rows),
        "last_match_date": rows["utcDate"].max() if len(rows) else None,
        "home_win_sum": rows.loc[rows["is_home"], "is_win"].sum(),
        "home_win_count": rows["is_home"].sum(),
        "away_win_sum": rows.loc[~rows["is_home"], "is_win"].sum(),
        "away_win_count": (~rows["is_home"]).sum(),
    }

    # Recompute streaks by walking results in order (mirrors compute_streak from Phase 4)
    streak_w, streak_l = 0, 0
    for r in rows["result"]:
        streak_w = streak_w + 1 if r == "WIN" else 0
        streak_l = streak_l + 1 if r == "LOSS" else 0
    state["win_streak"], state["loss_streak"] = streak_w, streak_l

    return state

all_teams = pd.unique(team_matches_season["team"])
team_state = {team: init_team_state(team) for team in all_teams}

# --- Global head-to-head state (all historical seasons, per Phase 4's non-season-scoped h2h) ---
h2h_state = defaultdict(lambda: {"win_sum": 0, "count": 0})

for _, m in clean_matches.sort_values("utcDate").iterrows():
    home, away = m["home_team_name"], m["away_team_name"]
    home_win = 1 if m["home_goals"] > m["away_goals"] else 0
    away_win = 1 if m["away_goals"] > m["home_goals"] else 0
    h2h_state[(home, away)]["win_sum"] += home_win
    h2h_state[(home, away)]["count"] += 1
    h2h_state[(away, home)]["win_sum"] += away_win
    h2h_state[(away, home)]["count"] += 1

In [20]:
import numpy as np

LEAGUE_AVG_GOALS = 1.4  # rough PL-wide average goals/team/match, used as fallback for early-season teams
SHRINKAGE_K = 5  # "games worth" of trust given to the league-average baseline

def get_feature_value(state, key_sum, key_count, default=LEAGUE_AVG_GOALS):
    """Safe average with a sane fallback when a team has little/no history yet."""
    count = state[key_count]
    if count == 0:
        return default
    return state[key_sum] / count

def shrunk_average(total_sum, count, baseline, k=SHRINKAGE_K):
    return (total_sum + k * baseline) / (count + k)

def expected_goals(home_state, away_state):
    home_attack = shrunk_average(home_state["goals_scored_sum"], home_state["goals_scored_count"], LEAGUE_AVG_GOALS)
    away_defense = shrunk_average(away_state["goals_conceded_sum"], away_state["goals_conceded_count"], LEAGUE_AVG_GOALS)
    away_attack = shrunk_average(away_state["goals_scored_sum"], away_state["goals_scored_count"], LEAGUE_AVG_GOALS)
    home_defense = shrunk_average(home_state["goals_conceded_sum"], home_state["goals_conceded_count"], LEAGUE_AVG_GOALS)

    home_lambda = (home_attack + away_defense) / 2
    away_lambda = (away_attack + home_defense) / 2

    home_lambda *= 1.1  # home advantage bump

    return max(home_lambda, 0.1), max(away_lambda, 0.1)


def sample_scoreline(home_state, away_state, target_outcome, max_attempts=200):
    """
    target_outcome: 'WIN', 'DRAW', or 'LOSS' from the HOME team's perspective.
    Resamples from Poisson(home_lambda)/Poisson(away_lambda) until the
    scoreline agrees with the classifier's chosen outcome.
    """
    home_lambda, away_lambda = expected_goals(home_state, away_state)

    for _ in range(max_attempts):
        home_goals = np.random.poisson(home_lambda)
        away_goals = np.random.poisson(away_lambda)

        if target_outcome == "WIN" and home_goals > away_goals:
            return home_goals, away_goals
        if target_outcome == "LOSS" and home_goals < away_goals:
            return home_goals, away_goals
        if target_outcome == "DRAW" and home_goals == away_goals:
            return home_goals, away_goals

    # Fallback: force a minimal valid scoreline for this outcome (extremely rare to hit this)
    if target_outcome == "WIN":
        return 1, 0
    if target_outcome == "LOSS":
        return 0, 1
    return 0, 0

In [26]:
FEATURE_ORDER = [
    "home_rolling_form_pts", "home_win_streak", "home_loss_streak",
    "home_avg_goals_scored", "home_avg_goals_conceded", "home_clean_sheet_pct",
    "home_days_since_last_match", "home_h2h_win_pct",
    "away_rolling_form_pts", "away_win_streak", "away_loss_streak",
    "away_avg_goals_scored", "away_avg_goals_conceded", "away_clean_sheet_pct",
    "away_days_since_last_match", "away_h2h_win_pct",
]

OUTCOME_MAP = {0: "LOSS", 1: "DRAW", 2: "WIN"}  # from the home team's perspective

def get_h2h_win_pct(h2h, home, away):
    rec = h2h[(home, away)]
    if rec["count"] == 0:
        return 0.5
    return rec["win_sum"] / rec["count"]


def build_feature_row(state, h2h, home, away, match_date):
    hs, as_ = state[home], state[away]

    def days_since(s):
        if s["last_match_date"] is None:
            return 7
        return (match_date - s["last_match_date"]).days

    return {
        "home_rolling_form_pts": sum(hs["recent_points"]),
        "home_win_streak": hs["win_streak"],
        "home_loss_streak": hs["loss_streak"],
        "home_avg_goals_scored": shrunk_average(hs["goals_scored_sum"], hs["goals_scored_count"], LEAGUE_AVG_GOALS),
        "home_avg_goals_conceded": shrunk_average(hs["goals_conceded_sum"], hs["goals_conceded_count"], LEAGUE_AVG_GOALS),
        "home_clean_sheet_pct": hs["clean_sheet_sum"] / hs["clean_sheet_count"] if hs["clean_sheet_count"] else 0.0,
        "home_days_since_last_match": days_since(hs),
        "home_h2h_win_pct": get_h2h_win_pct(h2h, home, away),

        "away_rolling_form_pts": sum(as_["recent_points"]),
        "away_win_streak": as_["win_streak"],
        "away_loss_streak": as_["loss_streak"],
        "away_avg_goals_scored": shrunk_average(as_["goals_scored_sum"], as_["goals_scored_count"], LEAGUE_AVG_GOALS),
        "away_avg_goals_conceded": shrunk_average(as_["goals_conceded_sum"], as_["goals_conceded_count"], LEAGUE_AVG_GOALS),
        "away_clean_sheet_pct": as_["clean_sheet_sum"] / as_["clean_sheet_count"] if as_["clean_sheet_count"] else 0.0,
        "away_days_since_last_match": days_since(as_),
        "away_h2h_win_pct": get_h2h_win_pct(h2h, away, home),
    }


def update_team_state(state, points, result, goals_scored, goals_conceded, is_home, match_date):
    state["recent_points"] = (state["recent_points"] + [points])[-5:]
    state["win_streak"] = state["win_streak"] + 1 if result == "WIN" else 0
    state["loss_streak"] = state["loss_streak"] + 1 if result == "LOSS" else 0
    state["goals_scored_sum"] += goals_scored
    state["goals_scored_count"] += 1
    state["goals_conceded_sum"] += goals_conceded
    state["goals_conceded_count"] += 1
    state["clean_sheet_sum"] += 1 if goals_conceded == 0 else 0
    state["clean_sheet_count"] += 1
    state["last_match_date"] = match_date
    if is_home:
        state["home_win_sum"] += 1 if result == "WIN" else 0
        state["home_win_count"] += 1
    else:
        state["away_win_sum"] += 1 if result == "WIN" else 0
        state["away_win_count"] += 1


def simulate_season(fixtures, initial_team_state, initial_h2h_state):
    state = {t: dict(s) for t, s in initial_team_state.items()}
    for s in state.values():
        s["recent_points"] = list(s["recent_points"])
    h2h = defaultdict(lambda: {"win_sum": 0, "count": 0},
                       {k: dict(v) for k, v in initial_h2h_state.items()})

    results = []

    for _, fx in fixtures.iterrows():
        home, away, match_date = fx["home_team"], fx["away_team"], fx["utcDate"]

        row = build_feature_row(state, h2h, home, away, match_date)
        X = pd.DataFrame([row])[FEATURE_ORDER]

        proba = model.predict_proba(X)[0]
        outcome_code = np.random.choice([0, 1, 2], p=proba)
        outcome = OUTCOME_MAP[outcome_code]

        home_goals, away_goals = sample_scoreline(state[home], state[away], outcome)

        home_points = {"WIN": 3, "DRAW": 1, "LOSS": 0}[outcome]
        away_result = {"WIN": "LOSS", "DRAW": "DRAW", "LOSS": "WIN"}[outcome]
        away_points = {"WIN": 3, "DRAW": 1, "LOSS": 0}[away_result]

        update_team_state(state[home], home_points, outcome, home_goals, away_goals, True, match_date)
        update_team_state(state[away], away_points, away_result, away_goals, home_goals, False, match_date)

        winner = home if outcome == "WIN" else away if outcome == "LOSS" else None
        loser = away if outcome == "WIN" else home if outcome == "LOSS" else None
        if winner:
            h2h[(winner, loser)]["win_sum"] += 1
        h2h[(home, away)]["count"] += 1
        h2h[(away, home)]["count"] += 1

        results.append({
            "match_id": fx["match_id"], "home_team": home, "away_team": away,
            "home_goals": home_goals, "away_goals": away_goals, "outcome": outcome,
        })

    return pd.DataFrame(results), state


# Re-run the test — team_state (the original) is now guaranteed untouched
test_results, final_state = simulate_season(remaining_fixtures, team_state, h2h_state)
print(test_results.shape)
print("Arsenal FC real recent_points (should be unchanged):", team_state["Arsenal FC"]["recent_points"])

(352, 6)
Arsenal FC real recent_points (should be unchanged): [3, 0, 3, 3, 1]


In [27]:
print(team_state["Arsenal FC"]["recent_points"])

[3, 0, 3, 3, 1]
